# Positive Definite Matrices & Quadratic Forms

正定矩阵与二次型。从二次型的矩阵表示到正定性判定，从 Cholesky 分解到条件数与优化，全程配代码与可视化。

## 0. 环境配置与导入

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
print("PyTorch version:", torch.__version__)
torch.manual_seed(42)

## 1. 二次型的定义与矩阵表示

**二次型**（quadratic form）是关于向量 $x \in \mathbb{R}^n$ 的二次齐次多项式：

$$
f(x) = \sum_{i=1}^n \sum_{j=1}^n a_{ij} x_i x_j
$$

可以用矩阵简洁表示为：

$$
f(x) = x^T A x
$$

其中 $A$ 是对称矩阵（非对称矩阵可以对称化：$A \to \frac{A+A^T}{2}$，不改变二次型的值）。

**2D 例子**：$f(x_1, x_2) = a x_1^2 + 2b x_1 x_2 + c x_2^2 = x^T \begin{pmatrix} a & b \\ b & c \end{pmatrix} x$

In [ ]:
# 二次型的矩阵表示
# 例子：f(x1, x2) = 2*x1^2 + 2*x1*x2 + 3*x2^2
A = torch.tensor([[2.0, 1.0],
                  [1.0, 3.0]])  # 对称矩阵

x = torch.tensor([1.0, 2.0])

# 直接计算
f_direct = 2*x[0]**2 + 2*x[0]*x[1] + 3*x[1]**2
# 矩阵形式
f_matrix = x.T @ A @ x

print("A =\n", A)
print("x =", x.tolist())
print(f"\n直接计算: f = 2*1^2 + 2*1*2 + 3*2^2 = {f_direct.item()}")
print(f"矩阵形式: f = x^T A x = {f_matrix.item()}")
print(f"相等? {torch.isclose(f_direct, f_matrix)}")

# 非对称矩阵的对称化
B = torch.tensor([[2.0, 3.0],
                  [-1.0, 3.0]])  # 非对称
B_sym = (B + B.T) / 2
print(f"\n非对称矩阵 B =\n{B}")
print(f"对称化 B_sym = (B+B^T)/2 =\n{B_sym}")
print(f"x^T B x = {(x.T @ B @ x).item()}")
print(f"x^T B_sym x = {(x.T @ B_sym @ x).item()}")
print(f"二次型值相同? {torch.isclose(x.T @ B @ x, x.T @ B_sym @ x)}")

## 2. 正定/半正定/负定/不定的分类

对于对称矩阵 A 和二次型 $f(x) = x^T A x$：

| 类型 | 定义 | 特征值条件 |
|------|------|-----------|
| **正定** (positive definite) | 对所有 $x \neq 0$，$x^T A x > 0$ | 所有特征值 $> 0$ |
| **半正定** (positive semidefinite) | 对所有 $x$，$x^T A x \geq 0$ | 所有特征值 $\geq 0$ |
| **负定** (negative definite) | 对所有 $x \neq 0$，$x^T A x < 0$ | 所有特征值 $< 0$ |
| **半负定** | 对所有 $x$，$x^T A x \leq 0$ | 所有特征值 $\leq 0$ |
| **不定** (indefinite) | 存在 $x$ 使 $x^T A x > 0$，也存在使 $< 0$ | 特征值有正有负 |

**正定矩阵的记号**：$A \succ 0$ 表示正定，$A \succeq 0$ 表示半正定。

In [ ]:
# 五种类型的矩阵例子
examples = {
    "正定": torch.tensor([[2.0, 0.0], [0.0, 3.0]]),
    "半正定": torch.tensor([[2.0, 0.0], [0.0, 0.0]]),
    "负定": torch.tensor([[-2.0, 0.0], [0.0, -3.0]]),
    "半负定": torch.tensor([[-2.0, 0.0], [0.0, 0.0]]),
    "不定": torch.tensor([[2.0, 0.0], [0.0, -3.0]]),
}

print(f"{'类型':>8} {'特征值':>20} {'x^T A x (x=[1,1])':>20}")
print("-" * 55)
for name, A in examples.items():
    eigvals = torch.linalg.eigvalsh(A)
    val = (torch.tensor([1.0, 1.0]).T @ A @ torch.tensor([1.0, 1.0])).item()
    print(f"{name:>8} {str(eigvals.tolist()):>20} {val:>20.1f}")

# 验证正定：对多个随机 x，x^T A x > 0
A_pos = examples["正定"]
print("\n验证正定性（1000 个随机 x）:")
min_val = float('inf')
for _ in range(1000):
    x = torch.randn(2)
    if torch.norm(x) > 0.01:
        val = (x.T @ A_pos @ x).item()
        min_val = min(min_val, val)
print(f"  min x^T A x = {min_val:.6f} > 0 → 正定")

# 不定矩阵：有正有负
A_ind = examples["不定"]
x_pos = torch.tensor([1.0, 0.0])
x_neg = torch.tensor([0.0, 1.0])
print(f"\n不定矩阵: x=[1,0] → x^T A x = {(x_pos.T @ A_ind @ x_pos).item()} > 0")
print(f"            x=[0,1] → x^T A x = {(x_neg.T @ A_ind @ x_neg).item()} < 0")

## 3. 正定性的三种判定方法

### 方法 1：特征值判定（最通用）

对称矩阵 A 正定 $\iff$ 所有特征值 $> 0$。

### 方法 2：顺序主子式判定（Sylvester 准则）

对称矩阵 A 正定 $\iff$ 所有顺序主子式 $> 0$。

顺序主子式 $\Delta_k$ 是 A 的左上角 $k \times k$ 子矩阵的行列式。

### 方法 3：Cholesky 分解判定

对称矩阵 A 正定 $\iff$ Cholesky 分解 $A = LL^T$ 存在且 L 的对角线元素全为正。

实际中 Cholesky 分解是最快的判定方法（$O(n^3/3)$，比特征值快）。

In [ ]:
# 三种判定方法对比
A = torch.tensor([[4.0, 2.0, 1.0],
                  [2.0, 5.0, 2.0],
                  [1.0, 2.0, 6.0]])

print("A =\n", A)
print("A 对称?", torch.allclose(A, A.T))

# 方法1：特征值
eigvals = torch.linalg.eigvalsh(A)
print(f"\n方法1 - 特征值判定:")
print(f"  特征值: {eigvals.tolist()}")
print(f"  全部 > 0? {(eigvals > 0).all().item()} → {'正定' if (eigvals>0).all() else '非正定'}")

# 方法2：顺序主子式
print(f"\n方法2 - 顺序主子式判定:")
all_pos = True
for k in range(1, 4):
    minor = A[:k, :k]
    det_minor = torch.det(minor).item()
    if det_minor <= 0:
        all_pos = False
    print(f"  Δ{k} = det(A[:{k},:{k}]) = {det_minor:.4f} {'(>0)' if det_minor > 0 else '(<=0!)'}")
print(f"  全部 > 0? {all_pos} → {'正定' if all_pos else '非正定'}")

# 方法3：Cholesky 分解
print(f"\n方法3 - Cholesky 分解判定:")
try:
    L = torch.linalg.cholesky(A)
    print(f"  Cholesky 分解成功!")
    print(f"  L =\n{L}")
    print(f"  L 对角线全正? {(L.diag() > 0).all().item()}")
    print(f"  验证 L@L^T = A? {torch.allclose(L @ L.T, A)}")
    print(f"  → 正定")
except RuntimeError as e:
    print(f"  Cholesky 分解失败: {e}")
    print(f"  → 非正定")

In [ ]:
# 非正定矩阵的判定对比
A_nonpd = torch.tensor([[1.0, 2.0, 3.0],
                         [2.0, 4.0, 5.0],
                         [3.0, 5.0, 6.0]])  # 这个矩阵不定

print("A =\n", A_nonpd)
eigvals = torch.linalg.eigvalsh(A_nonpd)
print(f"\n特征值: {eigvals.tolist()}")
print(f"有正有负 → 不定矩阵")

# Cholesky 会报错
print("\nCholesky 分解:")
try:
    L = torch.linalg.cholesky(A_nonpd)
    print("  成功（正定）")
except RuntimeError as e:
    print(f"  失败: {str(e)[:80]}")
    print("  → 非正定")

# 半正定矩阵（B @ B^T 一定半正定）
B = torch.randn(3, 2)
A_psd = B @ B.T
print(f"\n半正定矩阵 A = B@B^T (B 是 3x2):")
print(f"  特征值: {torch.linalg.eigvalsh(A_psd).tolist()}")
print(f"  全部 >= 0? {(torch.linalg.eigvalsh(A_psd) >= -1e-5).all().item()}")
print(f"  rank = {torch.linalg.matrix_rank(A_psd).item()} (< 3 → 半正定而非正定)")

## 4. 正定矩阵的性质

正定矩阵 A（$A \succ 0$）的重要性质：

1. **A 可逆**，且 $A^{-1} \succ 0$（逆矩阵也正定）
2. **A 的对角线元素全为正**：$A_{ii} = e_i^T A e_i > 0$
3. **$A$ 的所有主子式 $> 0$**（不只是顺序主子式）
4. **Cholesky 分解**：$A = LL^T$，L 对角线全正
5. **$A$ 可以开平方**：存在唯一正定矩阵 $A^{1/2}$，使得 $(A^{1/2})^2 = A$
6. **对任意可逆矩阵 C**，$C^T A C \succ 0$（合同变换保持正定性）
7. **$A$ 和 $B$ 都正定，则 $A+B$ 正定**
8. **$\det(A) > 0$**，且 $\det(A) \leq \prod A_{ii}$（Hadamard 不等式）

In [ ]:
# 正定矩阵性质验证
A = torch.tensor([[4.0, 1.0], [1.0, 3.0]])
B = torch.tensor([[2.0, 0.5], [0.5, 2.0]])

print("A =\n", A, "\nB =\n", B)
print("A 正定?", (torch.linalg.eigvalsh(A) > 0).all().item())
print("B 正定?", (torch.linalg.eigvalsh(B) > 0).all().item())

# 性质1：逆矩阵也正定
A_inv = torch.linalg.inv(A)
print(f"\n1. A^-1 的特征值: {torch.linalg.eigvalsh(A_inv).tolist()} (全正 → A^-1 正定)")

# 性质2：对角线元素全正
print(f"2. A 的对角线: {A.diag().tolist()} (全正)")

# 性质4：Cholesky 分解
L = torch.linalg.cholesky(A)
print(f"\n4. Cholesky: A = L@L^T? {torch.allclose(L@L.T, A)}")
print(f"   L 对角线: {L.diag().tolist()} (全正)")

# 性质5：矩阵平方根 A^(1/2)
# A = Q Λ Q^T, A^(1/2) = Q Λ^(1/2) Q^T
eigvals, Q = torch.linalg.eigh(A)
A_sqrt = Q @ torch.diag(torch.sqrt(eigvals)) @ Q.T
print(f"\n5. A^(1/2) =\n{A_sqrt}")
print(f"   (A^(1/2))^2 = A? {torch.allclose(A_sqrt @ A_sqrt, A, atol=1e-5)}")
print(f"   A^(1/2) 正定? {(torch.linalg.eigvalsh(A_sqrt) > 0).all().item()}")

# 性质6：合同变换保持正定性
C = torch.randn(2, 2)
C_AC = C.T @ A @ C
print(f"\n6. C^T A C 的特征值: {torch.linalg.eigvalsh(C_AC).tolist()} (全正 → 合同变换保持正定)")

# 性质7：A+B 正定
print(f"\n7. A+B 的特征值: {torch.linalg.eigvalsh(A+B).tolist()} (全正 → A+B 正定)")

# 性质8：det(A) > 0 且 det(A) <= prod(A_ii)
print(f"\n8. det(A) = {torch.det(A).item():.4f} > 0")
print(f"   prod(A_ii) = {A.diag().prod().item():.4f}")
print(f"   det(A) <= prod(A_ii)? {torch.det(A).item() <= A.diag().prod().item() + 1e-5} (Hadamard不等式)")

## 5. Cholesky 分解深化

Cholesky 分解将正定矩阵 A 分解为 $A = LL^T$，其中 L 是下三角矩阵，对角线元素为正。

**为什么重要**：
- 比 LU 分解快约 2 倍（利用对称性，只需处理半个矩阵）
- 数值稳定（不需要选主元，因为正定保证了稳定性）
- 是判定正定性的最快方法
- 广泛用于优化（牛顿法）、统计（高斯分布采样）、信号处理

**Cholesky 分解的递归理解**：
$$
A = \begin{pmatrix} a_{11} & b^T \\ b & C \end{pmatrix} = \begin{pmatrix} \sqrt{a_{11}} & 0 \\ b/\sqrt{a_{11}} & I \end{pmatrix} \begin{pmatrix} 1 & 0 \\ 0 & C - bb^T/a_{11} \end{pmatrix} \begin{pmatrix} \sqrt{a_{11}} & b/\sqrt{a_{11}} \\ 0 & I \end{pmatrix}
$$

其中 $C - bb^T/a_{11}$ 称为 Schur 补，递归分解即可。

In [ ]:
# 手动实现 Cholesky 分解（递归理解）
def cholesky_manual(A):
    """手动实现 Cholesky 分解，返回下三角矩阵 L"""
    n = A.shape[0]
    L = torch.zeros(n, n)
    for i in range(n):
        for j in range(i + 1):
            if i == j:
                # 对角线元素：L[i,i] = sqrt(A[i,i] - sum(L[i,:i]^2))
                L[i, i] = torch.sqrt(A[i, i] - torch.sum(L[i, :i] ** 2))
            else:
                # 非对角线：L[i,j] = (A[i,j] - sum(L[i,:j]*L[j,:j])) / L[j,j]
                L[i, j] = (A[i, j] - torch.sum(L[i, :j] * L[j, :j])) / L[j, j]
    return L

A = torch.tensor([[4.0, 2.0, 1.0],
                  [2.0, 5.0, 2.0],
                  [1.0, 2.0, 6.0]])

L_manual = cholesky_manual(A)
L_torch = torch.linalg.cholesky(A)

print("A =\n", A)
print("\n手动 Cholesky L =\n", L_manual)
print("\nPyTorch Cholesky L =\n", L_torch)
print("\n两者一致?", torch.allclose(L_manual, L_torch, atol=1e-5))
print("验证 L@L^T = A?", torch.allclose(L_manual @ L_manual.T, A, atol=1e-5))
print("L 下三角?", torch.allclose(L_manual, torch.tril(L_manual)))
print("L 对角线全正?", (L_manual.diag() > 0).all().item())

In [ ]:
# Cholesky 分解的应用：从多元高斯分布采样
# 若 X ~ N(0, I)，则 L @ X ~ N(0, Σ)，其中 Σ = L @ L^T
torch.manual_seed(42)

# 协方差矩阵（正定）
Sigma = torch.tensor([[1.0, 0.8],
                       [0.8, 1.0]])  # 强正相关
mu = torch.tensor([0.0, 0.0])

# Cholesky 分解
L = torch.linalg.cholesky(Sigma)
print("协方差矩阵 Σ =\n", Sigma)
print("Cholesky L =\n", L)
print("验证 L@L^T = Σ?", torch.allclose(L @ L.T, Sigma))

# 从标准正态采样，然后变换
n_samples = 1000
X_std = torch.randn(n_samples, 2)  # N(0, I)
X_transformed = X_std @ L.T + mu   # N(mu, Σ)

print(f"\n采样 {n_samples} 个点:")
print(f"  样本均值: {X_transformed.mean(dim=0).tolist()} (≈ [0,0])")
print(f"  样本协方差:\n{torch.cov(X_transformed.T)}")
print(f"  (≈ Σ = [[1,0.8],[0.8,1]])")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X_std[:, 0], X_std[:, 1], alpha=0.3, s=10, c='blue')
axes[0].set_title("标准正态 N(0, I)")
axes[0].set_aspect('equal'); axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(-4, 4); axes[0].set_ylim(-4, 4)

axes[1].scatter(X_transformed[:, 0], X_transformed[:, 1], alpha=0.3, s=10, c='red')
axes[1].set_title("变换后 N(0, Σ), Σ=[[1,0.8],[0.8,1]]")
axes[1].set_aspect('equal'); axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(-4, 4); axes[1].set_ylim(-4, 4)

plt.suptitle("Cholesky 分解用于高斯分布采样：X = L @ Z + μ", fontsize=13)
plt.tight_layout()
plt.show()

## 6. 二次型的几何意义

二次型 $f(x) = x^T A x$ 的等高线 $x^T A x = c$ 的形状由 A 的特征值决定：

- **正定**（特征值全正）：等高线是**椭圆**（n=2）或椭球（n=3）
- **负定**（特征值全负）：$x^T A x = c < 0$ 也是椭圆
- **不定**（有正有负）：等高线是**双曲线**，原点是**鞍点**
- **半正定**（有零特征值）：等高线是**抛物线**或退化的椭圆

椭圆的轴方向 = 特征向量方向，轴长度 = $\sqrt{c/\lambda_i}$。

In [ ]:
# 二次型的等高线可视化
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

x1 = torch.linspace(-3, 3, 200)
x2 = torch.linspace(-3, 3, 200)
X1, X2 = torch.meshgrid(x1, x2, indexing='ij')

matrices = {
    "正定: [[2,0],[0,1]]": torch.tensor([[2.0, 0.0], [0.0, 1.0]]),
    "不定: [[2,0],[0,-1]]": torch.tensor([[2.0, 0.0], [0.0, -1.0]]),
    "半正定: [[1,1],[1,1]]": torch.tensor([[1.0, 1.0], [1.0, 1.0]]),
}

for idx, (title, A) in enumerate(matrices.items()):
    ax = axes[idx]
    # 计算二次型值
    F = A[0,0]*X1**2 + 2*A[0,1]*X1*X2 + A[1,1]*X2**2
    # 等高线
    levels = torch.linspace(F.min(), F.max(), 15)
    cs = ax.contour(X1.numpy(), X2.numpy(), F.numpy(), levels=levels.numpy(), cmap='RdYlBu')
    ax.clabel(cs, inline=True, fontsize=7)
    # 特征向量方向
    eigvals, eigvecs = torch.linalg.eigh(A)
    for i in range(2):
        if abs(eigvals[i]) > 0.01:
            v = eigvecs[:, i]
            ax.arrow(0, 0, v[0]*2, v[1]*2, head_width=0.15, color='black', linewidth=1.5)
            ax.text(v[0]*2.2, v[1]*2.2, f'λ={eigvals[i]:.1f}', fontsize=9)
    ax.set_title(title, fontsize=11)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)

plt.suptitle("二次型 x^T A x 的等高线：椭圆(正定) / 双曲线(不定) / 抛物线(半正定)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# 3D 曲面图：正定=碗状(极小值)，不定=鞍状
fig = plt.figure(figsize=(14, 5))

x1 = torch.linspace(-2, 2, 100)
x2 = torch.linspace(-2, 2, 100)
X1, X2 = torch.meshgrid(x1, x2, indexing='ij')

# 正定：碗状
A_pos = torch.tensor([[1.0, 0.5], [0.5, 1.0]])
F_pos = A_pos[0,0]*X1**2 + 2*A_pos[0,1]*X1*X2 + A_pos[1,1]*X2**2
ax1 = fig.add_subplot(131, projection='3d')
ax1.plot_surface(X1.numpy(), X2.numpy(), F_pos.numpy(), cmap='viridis', alpha=0.8)
ax1.set_title("正定: 碗状 (极小值在原点)")
ax1.set_xlabel('x1'); ax1.set_ylabel('x2'); ax1.set_zlabel('f(x)')

# 不定：鞍状
A_ind = torch.tensor([[1.0, 0.0], [0.0, -1.0]])
F_ind = A_ind[0,0]*X1**2 + 2*A_ind[0,1]*X1*X2 + A_ind[1,1]*X2**2
ax2 = fig.add_subplot(132, projection='3d')
ax2.plot_surface(X1.numpy(), X2.numpy(), F_ind.numpy(), cmap='RdYlBu', alpha=0.8)
ax2.set_title("不定: 鞍状 (鞍点在原点)")
ax2.set_xlabel('x1'); ax2.set_ylabel('x2'); ax2.set_zlabel('f(x)')

# 负定：倒碗状
A_neg = torch.tensor([[-1.0, -0.5], [-0.5, -1.0]])
F_neg = A_neg[0,0]*X1**2 + 2*A_neg[0,1]*X1*X2 + A_neg[1,1]*X2**2
ax3 = fig.add_subplot(133, projection='3d')
ax3.plot_surface(X1.numpy(), X2.numpy(), F_neg.numpy(), cmap='inferno', alpha=0.8)
ax3.set_title("负定: 倒碗状 (极大值在原点)")
ax3.set_xlabel('x1'); ax3.set_ylabel('x2'); ax3.set_zlabel('f(x)')

plt.suptitle("二次型的 3D 曲面：正定性决定极值类型", fontsize=13)
plt.tight_layout()
plt.show()

## 7. 条件数与梯度下降

### 7.1 条件数

正定矩阵 A 的**条件数**（condition number）定义为：

$$
\kappa(A) = \frac{\lambda_{\max}}{\lambda_{\min}}
$$

条件数衡量矩阵的"病态"程度：
- $\kappa \approx 1$：良态（椭圆接近圆，各方向缩放均匀）
- $\kappa \gg 1$：病态（椭圆非常扁，各方向缩放差异大）

### 7.2 与梯度下降的关系

对于二次型 $f(x) = \frac{1}{2}x^T A x - b^T x$，梯度下降的收敛速度由条件数决定：

$$
\text{收敛率} \approx \left(\frac{\kappa - 1}{\kappa + 1}\right)^k
$$

- 条件数小（$\kappa \approx 1$）：收敛快，路径接近直线
- 条件数大（$\kappa \gg 1$）：收敛慢，路径呈锯齿状（Zigzag）

这就是为什么深度学习中要做**归一化/标准化**——降低损失函数 Hessian 矩阵的条件数，加速收敛。

In [ ]:
# 条件数对梯度下降收敛速度的影响
def gradient_descent_quadratic(A, b, x0, lr, n_steps):
    """二次型 f(x) = 0.5*x^T A x - b^T x 的梯度下降"""
    x = x0.clone()
    history = [x.clone()]
    for _ in range(n_steps):
        grad = A @ x - b  # df/dx = Ax - b
        x = x - lr * grad
        history.append(x.clone())
    return torch.stack(history)

# 两个矩阵：一个条件数小，一个条件数大
A_good = torch.tensor([[1.0, 0.0], [0.0, 1.5]])  # κ = 1.5
A_bad = torch.tensor([[1.0, 0.9], [0.9, 1.0]])    # κ = 19

b = torch.tensor([0.0, 0.0])  # 极小值在原点
x0 = torch.tensor([2.0, 1.0])

cond_good = torch.linalg.cond(A_good).item()
cond_bad = torch.linalg.cond(A_bad).item()
print(f"良态矩阵条件数 κ = {cond_good:.2f}")
print(f"病态矩阵条件数 κ = {cond_bad:.2f}")

# 梯度下降
lr = 0.3
n_steps = 50
hist_good = gradient_descent_quadratic(A_good, b, x0, lr, n_steps)
hist_bad = gradient_descent_quadratic(A_bad, b, x0, lr, n_steps)

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x1 = torch.linspace(-2.5, 2.5, 100)
x2 = torch.linspace(-2.5, 2.5, 100)
X1, X2 = torch.meshgrid(x1, x2, indexing='ij')

for idx, (A, hist, title, cond) in enumerate([
    (A_good, hist_good, f"良态 κ={cond_good:.1f}", cond_good),
    (A_bad, hist_bad, f"病态 κ={cond_bad:.1f}", cond_bad)
]):
    ax = axes[idx]
    F = 0.5 * (A[0,0]*X1**2 + 2*A[0,1]*X1*X2 + A[1,1]*X2**2)
    cs = ax.contour(X1.numpy(), X2.numpy(), F.numpy(), levels=15, cmap='RdYlBu')
    ax.plot(hist[:, 0].numpy(), hist[:, 1].numpy(), 'ko-', markersize=3, linewidth=1, alpha=0.7)
    ax.scatter([0], [0], c='red', s=100, marker='*', zorder=5, label='极小值')
    ax.set_title(f"{title}\n梯度下降{('快(接近直线)' if cond < 5 else '慢(锯齿状)')}")
    ax.set_aspect('equal'); ax.legend(); ax.grid(True, alpha=0.3)
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)

plt.suptitle("条件数对梯度下降收敛速度的影响", fontsize=13)
plt.tight_layout()
plt.show()

# 收敛曲线
fig, ax = plt.subplots(figsize=(8, 5))
error_good = torch.norm(hist_good, dim=1)
error_bad = torch.norm(hist_bad, dim=1)
ax.plot(error_good.numpy(), 'b-', linewidth=2, label=f'良态 κ={cond_good:.1f}')
ax.plot(error_bad.numpy(), 'r-', linewidth=2, label=f'病态 κ={cond_bad:.1f}')
ax.set_xlabel("迭代步数"); ax.set_ylabel("||x - x*|| (到极小值的距离)")
ax.set_title("梯度下降收敛曲线：条件数越大，收敛越慢")
ax.legend(); ax.grid(True, alpha=0.3); ax.set_yscale('log')
plt.tight_layout()
plt.show()

## 8. 应用：马氏距离与多元高斯分布

### 8.1 马氏距离

欧氏距离 $\|x - y\|_2$ 假设各维度独立且方差相同。当数据存在相关性和不同方差时，**马氏距离**（Mahalanobis distance）更合适：

$$
d_M(x, y; \Sigma) = \sqrt{(x-y)^T \Sigma^{-1} (x-y)}
$$

其中 $\Sigma$ 是协方差矩阵（正定）。

马氏距离的几何意义：在经过 $\Sigma^{-1/2}$ 变换后的空间中计算欧氏距离。它自动考虑了各维度的方差和相关性。

### 8.2 多元高斯分布

多元高斯分布的概率密度：

$$
p(x; \mu, \Sigma) = \frac{1}{(2\pi)^{n/2} |\Sigma|^{1/2}} \exp\left(-\frac{1}{2}(x-\mu)^T \Sigma^{-1} (x-\mu)\right)
$$

等高线 $(x-\mu)^T \Sigma^{-1} (x-\mu) = c$ 是椭圆，轴方向 = $\Sigma$ 的特征向量，轴长度 = $\sqrt{c \lambda_i}$。

In [ ]:
# 马氏距离 vs 欧氏距离
# 构造一个有相关性的数据集
torch.manual_seed(42)
n = 200
# 真实协方差：x 和 y 强正相关
Sigma = torch.tensor([[1.0, 0.9], [0.9, 1.0]])
L = torch.linalg.cholesky(Sigma)
data = torch.randn(n, 2) @ L.T  # N(0, Σ)

# 两个测试点
mu = torch.tensor([0.0, 0.0])
p1 = torch.tensor([2.0, 0.0])   # 沿 x 轴（方差大的方向）
p2 = torch.tensor([0.0, 2.0])   # 沿 y 轴（方差大的方向）
p3 = torch.tensor([1.5, 1.5])   # 沿相关方向（数据密集方向）

Sigma_inv = torch.linalg.inv(Sigma)

def mahalanobis(x, mu, Sigma_inv):
    diff = x - mu
    return torch.sqrt(diff @ Sigma_inv @ diff)

print("协方差矩阵 Σ =\n", Sigma)
print("\n各点到原点的距离:")
print(f"{'点':>12} {'欧氏距离':>10} {'马氏距离':>10} {'解释':>20}")
for name, p in [("(2,0) 沿x轴", p1), ("(0,2) 沿y轴", p2), ("(1.5,1.5) 相关方向", p3)]:
    d_euc = torch.norm(p - mu).item()
    d_mah = mahalanobis(p, mu, Sigma_inv).item()
    explain = "异常" if d_mah > 2 else ("正常" if d_mah < 1 else "边缘")
    print(f"{name:>12} {d_euc:>10.3f} {d_mah:>10.3f} {explain:>20}")

print("\n→ 马氏距离考虑了数据的相关性：沿相关方向(1.5,1.5)虽然欧氏距离大，但在数据分布方向上，马氏距离小")
print("→ 沿垂直于相关方向的点，虽然欧氏距离可能小，但马氏距离大（异常点）")

# 可视化
fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(data[:, 0], data[:, 1], alpha=0.3, s=20, c='blue', label='数据 N(0,Σ)')
for name, p, color in [("p1=(2,0)", p1, 'red'), ("p2=(0,2)", p2, 'green'), ("p3=(1.5,1.5)", p3, 'orange')]:
    ax.scatter(p[0], p[1], c=color, s=150, marker='*', zorder=5, label=name)
# 马氏距离等高线
x1 = torch.linspace(-3, 3, 100)
x2 = torch.linspace(-3, 3, 100)
X1, X2 = torch.meshgrid(x1, x2, indexing='ij')
mah = torch.zeros_like(X1)
for i in range(100):
    for j in range(100):
        diff = torch.tensor([X1[i,j], X2[i,j]])
        mah[i,j] = torch.sqrt(diff @ Sigma_inv @ diff)
cs = ax.contour(X1.numpy(), X2.numpy(), mah.numpy(), levels=[1, 2, 3], colors='black', linestyles='--')
ax.clabel(cs, inline=True, fontsize=9)
ax.set_title("马氏距离等高线（椭圆）与数据分布")
ax.set_aspect('equal'); ax.legend(); ax.grid(True, alpha=0.3)
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
plt.tight_layout()
plt.show()

## 课后练习

### 基础题

1. 将二次型 $f(x_1, x_2, x_3) = x_1^2 + 2x_2^2 + 3x_3^2 + 2x_1x_2 - 2x_1x_3 + 4x_2x_3$ 写成矩阵形式 $x^T A x$，验证矩阵是对称的。

2. 判断下列矩阵的类型（正定/半正定/负定/半负定/不定），用特征值验证：
   - $\begin{pmatrix} 2 & 1 \\ 1 & 2 \end{pmatrix}$
   - $\begin{pmatrix} 1 & 2 \\ 2 & 4 \end{pmatrix}$
   - $\begin{pmatrix} -1 & 0 \\ 0 & -2 \end{pmatrix}$
   - $\begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$

3. 用三种方法（特征值、顺序主子式、Cholesky）判定矩阵 $A = \begin{pmatrix} 5 & 2 & 1 \\ 2 & 4 & 2 \\ 1 & 2 & 3 \end{pmatrix}$ 是否正定。

### 正定矩阵性质

4. 验证正定矩阵的性质：取一个 3×3 正定矩阵 A，验证：
   - $A^{-1}$ 也正定
   - $A$ 的对角线元素全正
   - $A$ 可以开平方 $A^{1/2}$，且 $(A^{1/2})^2 = A$
   - 对任意可逆矩阵 C，$C^T A C$ 正定

5. 手动实现 Cholesky 分解（不调用 `torch.linalg.cholesky`），对一个 4×4 正定矩阵验证结果与 PyTorch 一致。

### 几何与优化

6. 画出二次型 $f(x) = x^T \begin{pmatrix} 3 & 1 \\ 1 & 2 \end{pmatrix} x$ 的等高线图，在图上标出特征向量方向，验证椭圆的轴就是特征向量方向。

7. 梯度下降实验：构造两个 2D 二次型，一个条件数 $\kappa=2$，一个 $\kappa=50$，用相同学习率做梯度下降，画出收敛路径和收敛曲线，解释条件数对收敛速度的影响。

### 应用题

8. 马氏距离：生成一个 2D 数据集（协方差 $\Sigma = \begin{pmatrix} 1 & 0.8 \\ 0.8 & 1 \end{pmatrix}$），取三个测试点，分别计算欧氏距离和马氏距离到均值的距离，解释为什么马氏距离更适合异常检测。

9. 多元高斯采样：用 Cholesky 分解从协方差为 $\Sigma = \begin{pmatrix} 4 & 1 \\ 1 & 1 \end{pmatrix}$ 的二元高斯分布中采样 1000 个点，验证样本协方差接近 $\Sigma$，并画出散点图和马氏距离等高线。

### 综合题

10. 证明：如果 A 正定，则对任意非零向量 x，$x^T A x > 0$。用特征分解 $A = Q \Lambda Q^T$ 证明。

11. 解释为什么在深度学习中，对输入数据做标准化（减去均值除以标准差）可以加速梯度下降。提示：从 Hessian 矩阵条件数的角度解释。